# LeetCode 3740 — Minimum Distance Between Three Equal Elements I bằng Python

Link đề: https://leetcode.com/problems/minimum-distance-between-three-equal-elements-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def minimumDistance(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org. (Bài II cùng ý tưởng nhưng constraints lớn hơn.)

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm ba chỉ số **khác nhau** `i, j, k` sao cho `nums[i] == nums[j] == nums[k]` (một "bộ ba tốt") và **khoảng cách** `|i-j| + |j-k| + |k-i|` nhỏ nhất.
2. Rút gọn công thức khoảng cách: với `i < j < k` thì khoảng cách bằng `2 * (k - i)` — chỉ phụ thuộc hai chỉ số **ngoài cùng**.
3. Biết gom các chỉ số theo giá trị bằng `dict` và dùng **cửa sổ ba phần tử liên tiếp** trong danh sách chỉ số.
4. Viết 4 cách: vét cạn ba vòng lặp, gom nhóm theo giá trị, một lượt với "hai vị trí gần nhất", và sắp xếp theo `(giá trị, chỉ số)`; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: bỏ sót số hạng `|k-i|`, chỉ lấy ba lần xuất hiện đầu tiên, trả `inf`/`0` thay vì `-1`, và quên điều kiện "giá trị bằng nhau".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Một bộ ba chỉ số **đôi một khác nhau** `(i, j, k)` gọi là **tốt** nếu `nums[i] == nums[j] == nums[k]`. **Khoảng cách** của bộ ba tốt là `|i - j| + |j - k| + |k - i|`.

Trả về khoảng cách **nhỏ nhất** của một bộ ba tốt; nếu không tồn tại bộ ba tốt thì trả về `-1`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,1,1,3]` | `6` | Bộ `(0, 2, 3)` đều có giá trị `1`: `2 + 1 + 3 = 6` |
| 2 | `[1,1,2,3,2,1,2]` | `8` | Bộ `(2, 4, 6)` đều có giá trị `2`: `2 + 2 + 4 = 8` |
| 3 | `[1]` | `-1` | Không có bộ ba tốt nào |

Constraints: `1 <= n == nums.length <= 100`, `1 <= nums[i] <= n`.

In [1]:
def distance(i, j, k):
    return abs(i - j) + abs(j - k) + abs(k - i)

print(distance(0, 2, 3), distance(2, 4, 6))
nums = [1, 2, 1, 1, 3]
print([(i, nums[i]) for i in range(len(nums))])
print("bộ ba (0,2,3):", [nums[0], nums[2], nums[3]], "khoảng cách", distance(0, 2, 3))

6 8
[(0, 1), (1, 2), (2, 1), (3, 1), (4, 3)]
bộ ba (0,2,3): [1, 1, 1] khoảng cách 6


## 3. Kiến thức nền

### 3.1. Trị tuyệt đối `abs` và khoảng cách giữa hai chỉ số

`abs(x)` là giá trị tuyệt đối. `|i - j|` là khoảng cách giữa hai vị trí trên trục chỉ số, **không phụ thuộc thứ tự** của `i` và `j`. Nếu đã biết `i < j` thì `|i - j| = j - i`, không cần `abs`.

In [2]:
print(abs(3 - 7), abs(7 - 3), abs(0))
i, j = 2, 5
print(abs(i - j) == j - i)

4 4 0
True


### 3.2. `dict` và `defaultdict(list)` để gom nhóm

`defaultdict(list)` tự tạo danh sách rỗng khi gặp khóa mới. Duyệt mảng và `pos[value].append(index)` sẽ gom **tất cả chỉ số xuất hiện** của từng giá trị — danh sách này **tăng dần** vì ta duyệt từ trái sang phải.

In [3]:
from collections import defaultdict
nums = [1, 1, 2, 3, 2, 1, 2]
pos = defaultdict(list)
for i, v in enumerate(nums):
    pos[v].append(i)
print(dict(pos))

{1: [0, 1, 5], 2: [2, 4, 6], 3: [3]}


### 3.3. Cửa sổ trượt kích thước 3 trên một danh sách

Với danh sách `p`, các **bộ ba liên tiếp** là `(p[t], p[t+1], p[t+2])` với `t = 0 .. len(p) - 3`. Nếu `len(p) < 3` thì không có bộ nào (`range(len(p) - 2)` rỗng, kể cả khi âm).

In [4]:
p = [0, 1, 4, 5, 6]
print([(p[t], p[t + 1], p[t + 2]) for t in range(len(p) - 2)])
print(list(range(len([4, 7]) - 2)))      # range(0) rỗng

[(0, 1, 4), (1, 4, 5), (4, 5, 6)]
[]


### 3.4. `min` với giá trị khởi tạo và `float('inf')`

Khi tìm "nhỏ nhất" ta khởi tạo `best = float('inf')` (lớn hơn mọi số) rồi `best = min(best, cand)`. Cuối cùng nhớ đổi `inf` thành `-1` nếu chưa có ứng viên nào.

In [5]:
best = float('inf')
for cand in [8, 4, 10]:
    best = min(best, cand)
print(best)
print(float('inf') > 10 ** 18, -1 if float('inf') == float('inf') else 0)

4
True -1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giả sử ba chỉ số đã sắp xếp `i < j < k`. Khi đó:

```
|i-j| + |j-k| + |k-i| = (j - i) + (k - j) + (k - i) = 2 * (k - i)
```

Chỉ số ở **giữa** `j` bị triệt tiêu! Khoảng cách chỉ phụ thuộc vào "độ dài đoạn" từ `i` đến `k`, và bộ ba nhỏ nhất là bộ có đoạn `[i, k]` **ngắn nhất** chứa đủ ba vị trí cùng giá trị.

Tính tay ví dụ 2 `[1,1,2,3,2,1,2]`: giá trị `1` ở chỉ số `0, 1, 5`; giá trị `2` ở `2, 4, 6`; giá trị `3` chỉ một lần.

```
1: (0,1,5) -> 2*(5-0) = 10
2: (2,4,6) -> 2*(6-2) = 8   <- nhỏ nhất
```

Trực giác thứ hai: với một giá trị có danh sách chỉ số `p` (tăng dần), để `k - i` nhỏ nhất ta nên chọn ba chỉ số **liên tiếp** trong `p` (xen thêm chỉ số khác vào giữa chỉ làm đoạn dài ra).

### Bảng tính tay

| nums | giá trị: danh sách chỉ số | các bộ ba liên tiếp | `2*(p[t+2]-p[t])` | đáp án |
|---|---|---|---|---|
| `[1,2,1,1,3]` | 1: [0,2,3] | (0,2,3) | 6 | 6 |
| `[1,1,2,3,2,1,2]` | 1: [0,1,5]; 2: [2,4,6] | (0,1,5); (2,4,6) | 10; 8 | 8 |
| `[1]` | 1: [0] | không có | - | -1 |
| `[1,1,1,1]` | 1: [0,1,2,3] | (0,1,2); (1,2,3) | 4; 4 | 4 |

In [6]:
from collections import defaultdict

def by_hand(nums):
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    best = None
    for v, p in pos.items():
        for t in range(len(p) - 2):
            d = 2 * (p[t + 2] - p[t])
            print(f"  giá trị {v}: bộ ({p[t]},{p[t + 1]},{p[t + 2]}) -> {d}")
            best = d if best is None else min(best, d)
    return -1 if best is None else best

for nums in [[1, 2, 1, 1, 3], [1, 1, 2, 3, 2, 1, 2], [1], [1, 1, 1, 1]]:
    print(nums, "=>", by_hand(nums))

  giá trị 1: bộ (0,2,3) -> 6
[1, 2, 1, 1, 3] => 6
  giá trị 1: bộ (0,1,5) -> 10
  giá trị 2: bộ (2,4,6) -> 8
[1, 1, 2, 3, 2, 1, 2] => 8
[1] => -1
  giá trị 1: bộ (0,1,2) -> 4
  giá trị 1: bộ (1,2,3) -> 4
[1, 1, 1, 1] => 4


## 5. Cách 1 — Brute Force / baseline

Làm đúng như định nghĩa: ba vòng lặp lồng `i < j < k`; nếu ba giá trị bằng nhau thì tính khoảng cách **theo công thức gốc** `abs(i-j) + abs(j-k) + abs(k-i)` và lấy min.

Nút thắt: `O(n^3)` bộ ba. Với `n <= 100` (khoảng `1.6` trăm nghìn bộ) vẫn chạy được, nhưng ta thấy ngay phần lớn bộ ba bị loại vì giá trị khác nhau, và nhiều bộ ba chứa chỉ số "thừa" ở giữa.

In [7]:
class SolutionBrute:
    def minimumDistance(self, nums):
        n = len(nums)
        best = float('inf')
        for i in range(n):
            for j in range(i + 1, n):
                if nums[j] != nums[i]:
                    continue
                for k in range(j + 1, n):
                    if nums[k] == nums[i]:
                        best = min(best, abs(i - j) + abs(j - k) + abs(k - i))
        return -1 if best == float('inf') else best

for nums in [[1, 2, 1, 1, 3], [1, 1, 2, 3, 2, 1, 2], [1]]:
    print(nums, "->", SolutionBrute().minimumDistance(nums))

[1, 2, 1, 1, 3] -> 6
[1, 1, 2, 3, 2, 1, 2] -> 8
[1] -> -1


**Độ phức tạp:** Time `O(n^3)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** khoảng cách `= 2 * (k - i)` với `i < j < k`; do đó chỉ cần xét các bộ ba **liên tiếp trong danh sách chỉ số của từng giá trị**.

**Quy tắc "nếu… thì…":**

- Nếu một giá trị xuất hiện **dưới 3 lần** -> nó không tạo được bộ ba nào, bỏ qua.
- Nếu danh sách chỉ số của một giá trị là `p` -> ứng viên là `2 * (p[t+2] - p[t])` với mọi `t`.
- Nếu không có ứng viên nào -> trả `-1`.

**Invariant (cách một lượt):** khi tới chỉ số `k` với giá trị `v`, nếu đã thấy `v` ít nhất hai lần trước đó ở các chỉ số gần nhất `i < j` thì bộ ba tốt nhất **kết thúc tại `k`** có chỉ số `i` là *chỉ số thứ hai tính ngược* — cho ứng viên `2 * (k - i)`.

In [8]:
# Kiểm chứng công thức 2*(k-i) với mọi bộ ba i<j<k
for i in range(8):
    for j in range(i + 1, 8):
        for k in range(j + 1, 8):
            assert distance(i, j, k) == 2 * (k - i)
print("|i-j| + |j-k| + |k-i| = 2*(k-i) khi i<j<k")

|i-j| + |j-k| + |k-i| = 2*(k-i) khi i<j<k


## 7. Các cách tiếp theo

### Cách 2 — Gom chỉ số theo giá trị rồi quét cửa sổ ba

Dùng `defaultdict(list)` để có `pos[v]`; với mỗi nhóm có ít nhất 3 phần tử, quét `t` từ `0` đến `len(p) - 3` và cập nhật `best = min(best, 2 * (p[t+2] - p[t]))`.

In [9]:
from collections import defaultdict

class SolutionGroup:
    def minimumDistance(self, nums):
        pos = defaultdict(list)
        for i, v in enumerate(nums):
            pos[v].append(i)
        best = float('inf')
        for p in pos.values():
            for t in range(len(p) - 2):
                best = min(best, 2 * (p[t + 2] - p[t]))
        return -1 if best == float('inf') else best

for nums in [[1, 2, 1, 1, 3], [1, 1, 2, 3, 2, 1, 2], [1]]:
    print(nums, "->", SolutionGroup().minimumDistance(nums))

[1, 2, 1, 1, 3] -> 6
[1, 1, 2, 3, 2, 1, 2] -> 8
[1] -> -1


**Độ phức tạp:** Time `O(n)` (mỗi chỉ số thuộc đúng một nhóm), Space `O(n)`.

### Cách 3 — Một lượt, chỉ nhớ hai vị trí gần nhất của mỗi giá trị

Không cần lưu cả danh sách. Với mỗi giá trị `v` ta giữ `last2[v] = (vị trí thứ hai tính ngược, vị trí gần nhất)`. Khi tới `k` có giá trị `v`: nếu `v` đã có hai vị trí `(a, b)` thì ứng viên `2 * (k - a)`; sau đó cập nhật `(a, b) = (b, k)`. Không gian chỉ `O(số giá trị khác nhau)` nhưng mỗi giá trị chỉ lưu hai số.

In [10]:
class SolutionOnePass:
    def minimumDistance(self, nums):
        last = {}                          # v -> [vị trí thứ hai tính ngược, vị trí gần nhất]
        best = float('inf')
        for k, v in enumerate(nums):
            if v in last:
                a, b = last[v]
                if a is not None:
                    best = min(best, 2 * (k - a))
                last[v] = [b, k]
            else:
                last[v] = [None, k]
        return -1 if best == float('inf') else best

for nums in [[1, 2, 1, 1, 3], [1, 1, 2, 3, 2, 1, 2], [1]]:
    print(nums, "->", SolutionOnePass().minimumDistance(nums))

[1, 2, 1, 1, 3] -> 6
[1, 1, 2, 3, 2, 1, 2] -> 8
[1] -> -1


**Độ phức tạp:** Time `O(n)`, Space `O(m)` với `m` là số giá trị phân biệt (`<= n`), mỗi giá trị hai số.

### Cách 4 — Sắp xếp theo `(giá trị, chỉ số)` rồi trượt cửa sổ ba

Sắp các chỉ số theo khóa `(nums[i], i)`. Các chỉ số cùng giá trị nằm liền nhau và đã tăng dần. Với ba chỉ số liên tiếp `a, b, c` trong thứ tự này: nếu `nums[a] == nums[c]` thì cả ba cùng giá trị (vì sắp theo giá trị) và ứng viên là `2 * (c - a)`.

In [11]:
class SolutionSort:
    def minimumDistance(self, nums):
        order = sorted(range(len(nums)), key=lambda i: (nums[i], i))
        best = float('inf')
        for t in range(len(order) - 2):
            a, c = order[t], order[t + 2]
            if nums[a] == nums[c]:
                best = min(best, 2 * (c - a))
        return -1 if best == float('inf') else best

for nums in [[1, 2, 1, 1, 3], [1, 1, 2, 3, 2, 1, 2], [1]]:
    print(nums, "->", SolutionSort().minimumDistance(nums))

[1, 2, 1, 1, 3] -> 6
[1, 1, 2, 3, 2, 1, 2] -> 8
[1] -> -1


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [1, 1, 2, 3, 2, 1, 2]`:

| k | v | `last[v]` trước | ứng viên | `best` | `last[v]` sau |
|---|---|---|---|---|---|
| 0 | 1 | (chưa có) | - | inf | [None, 0] |
| 1 | 1 | [None, 0] | - | inf | [0, 1] |
| 2 | 2 | (chưa có) | - | inf | [None, 2] |
| 3 | 3 | (chưa có) | - | inf | [None, 3] |
| 4 | 2 | [None, 2] | - | inf | [2, 4] |
| 5 | 1 | [0, 1] | 2*(5-0) = 10 | 10 | [1, 5] |
| 6 | 2 | [2, 4] | 2*(6-2) = 8 | 8 | [4, 6] |

In [12]:
def trace_one_pass(nums):
    print("nums =", nums)
    last, best = {}, float('inf')
    for k, v in enumerate(nums):
        prev = last.get(v)
        cand = None
        if prev is not None and prev[0] is not None:
            cand = 2 * (k - prev[0])
            best = min(best, cand)
        last[v] = [prev[1], k] if prev else [None, k]
        print(f"  k={k} v={v} trước={prev} ứng viên={cand} best={best} sau={last[v]}")
    return -1 if best == float('inf') else best

assert trace_one_pass([1, 1, 2, 3, 2, 1, 2]) == 8
assert trace_one_pass([1]) == -1

nums = [1, 1, 2, 3, 2, 1, 2]
  k=0 v=1 trước=None ứng viên=None best=inf sau=[None, 0]
  k=1 v=1 trước=[None, 0] ứng viên=None best=inf sau=[0, 1]
  k=2 v=2 trước=None ứng viên=None best=inf sau=[None, 2]
  k=3 v=3 trước=None ứng viên=None best=inf sau=[None, 3]
  k=4 v=2 trước=[None, 2] ứng viên=None best=inf sau=[2, 4]
  k=5 v=1 trước=[0, 1] ứng viên=10 best=10 sau=[1, 5]
  k=6 v=2 trước=[2, 4] ứng viên=8 best=8 sau=[4, 6]
nums = [1]
  k=0 v=1 trước=None ứng viên=None best=inf sau=[None, 0]


In [13]:
def trace_group(nums):
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    print("nhóm chỉ số:", dict(pos))
    best = float('inf')
    for v, p in pos.items():
        if len(p) < 3:
            print(f"  giá trị {v}: chỉ {len(p)} lần -> bỏ qua")
            continue
        for t in range(len(p) - 2):
            cand = 2 * (p[t + 2] - p[t])
            best = min(best, cand)
            print(f"  giá trị {v}: ({p[t]},{p[t + 1]},{p[t + 2]}) -> {cand}, best={best}")
    return -1 if best == float('inf') else best

assert trace_group([1, 2, 1, 1, 3]) == 6

nhóm chỉ số: {1: [0, 2, 3], 2: [1], 3: [4]}
  giá trị 1: (0,2,3) -> 6, best=6
  giá trị 2: chỉ 1 lần -> bỏ qua
  giá trị 3: chỉ 1 lần -> bỏ qua


## 9. Code nộp LeetCode

In [14]:
from typing import List, Optional
from collections import defaultdict

class Solution:
    def minimumDistance(self, nums: List[int]) -> int:
        pos = defaultdict(list)
        for i, v in enumerate(nums):
            pos[v].append(i)
        best = float('inf')
        for p in pos.values():
            for t in range(len(p) - 2):          # ba chỉ số liên tiếp trong danh sách vị trí
                best = min(best, 2 * (p[t + 2] - p[t]))
        return -1 if best == float('inf') else best

## 10. Vì sao thuật toán đúng

- **Công thức:** với `i < j < k`, `|i-j| + |j-k| + |k-i| = (j-i) + (k-j) + (k-i) = 2(k-i)`. Khoảng cách không phụ thuộc thứ tự gán `(i, j, k)` vì biểu thức đối xứng, nên có thể giả sử đã sắp xếp.
- **Chỉ cần bộ ba liên tiếp:** giả sử một bộ ba tốt của giá trị `v` có chỉ số trong danh sách `p` lần lượt là `p[a] < p[b] < p[c]` (với `a < b < c`). Bộ ba liên tiếp `(p[a], p[a+1], p[a+2])` cũng tốt và có `p[a+2] <= p[c]` nên khoảng cách **không lớn hơn**. Vậy tối ưu đạt được ở bộ ba liên tiếp.
- **Dưới 3 lần xuất hiện:** không đủ ba chỉ số khác nhau -> bị bỏ qua; nếu mọi giá trị đều như vậy thì không có ứng viên và trả `-1`.
- **Cách một lượt:** tại `k`, bộ ba tốt tốt nhất kết thúc ở `k` dùng hai vị trí gần nhất trước đó, tức vị trí thứ hai tính ngược `a`; ứng viên `2(k - a)`. Lấy min trên mọi `k` bao phủ mọi bộ ba liên tiếp.
- **Cách sắp xếp:** thứ tự `(giá trị, chỉ số)` làm các chỉ số cùng giá trị liền nhau và tăng dần, nên ba phần tử liên tiếp có `nums[a] == nums[c]` chính xác là bộ ba liên tiếp trong nhóm của giá trị đó.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Ba vòng lặp | `O(n^3)` | `O(1)` |
| Gom nhóm theo giá trị | `O(n)` | `O(n)` |
| Một lượt, hai vị trí gần nhất | `O(n)` | `O(m)`, mỗi giá trị 2 số |
| Sắp xếp `(giá trị, chỉ số)` | `O(n log n)` | `O(n)` |

`n = len(nums) <= 100`, `m` là số giá trị phân biệt. Phần Bài II của LeetCode có `n` lớn hơn nên lời giải `O(n)` ở đây là chuẩn.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `n = 2`, toàn bộ phần tử bằng nhau, đúng hai lần mỗi giá trị) và so sánh **tất cả cách** trên nhiều input hợp lệ (đảm bảo `1 <= nums[i] <= n`).

In [15]:
impls = [SolutionBrute(), SolutionGroup(), SolutionOnePass(), SolutionSort(), Solution()]

cases = [
    ([1, 2, 1, 1, 3], 6), ([1, 1, 2, 3, 2, 1, 2], 8), ([1], -1),
    ([1, 1], -1), ([1, 1, 1], 4), ([1, 1, 1, 1], 4), ([2, 1, 2, 1], -1),
    ([1, 2, 1, 2, 1, 2], 8), ([1, 1, 2, 2, 1, 1, 1], 4), ([3, 3, 3, 2, 2], 4),
    ([1] * 100, 4),
]
for nums, exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= len(nums) for v in nums)
    for impl in impls:
        got = impl.minimumDistance(list(nums))
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 11 ca


In [16]:
import random
random.seed(3740)
nonneg = 0
for _ in range(1500):
    n = random.randint(1, 30)
    hi = random.choice([n, min(n, 3), min(n, 5)])           # ép nhiều giá trị lặp
    nums = [random.randint(1, hi) for _ in range(n)]
    outs = [impl.minimumDistance(list(nums)) for impl in impls]
    assert all(o == outs[0] for o in outs), (nums, outs)
    nonneg += outs[0] != -1
print("1500 test ngẫu nhiên (seed=3740) OK; số ca có đáp án != -1:", nonneg)
assert nonneg > 500

1500 test ngẫu nhiên (seed=3740) OK; số ca có đáp án != -1: 1194


In [17]:
# Độ dài tối đa n = 100: so sánh nhanh các cách tuyến tính với brute
for _ in range(30):
    nums = [random.randint(1, random.choice([100, 10, 4])) for _ in range(100)]
    assert SolutionBrute().minimumDistance(nums) == Solution().minimumDistance(nums) == SolutionOnePass().minimumDistance(nums)
print("n = 100 OK")

n = 100 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên số hạng `|k - i|`

Chỉ cộng `|i-j| + |j-k|` ra `k - i` thay vì `2(k - i)`.

In [18]:
def wrong_two_terms(nums):
    best = float('inf')
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] == nums[j] == nums[k]:
                    best = min(best, abs(i - j) + abs(j - k))       # SAI: thiếu abs(k - i)
    return -1 if best == float('inf') else best

print(wrong_two_terms([1, 2, 1, 1, 3]), "!=", Solution().minimumDistance([1, 2, 1, 1, 3]))
assert wrong_two_terms([1, 2, 1, 1, 3]) == 3 and Solution().minimumDistance([1, 2, 1, 1, 3]) == 6

3 != 6


### Lỗi 2: chỉ lấy ba lần xuất hiện đầu tiên của mỗi giá trị

Bộ ba tốt nhất có thể nằm ở các lần xuất hiện **sau**: ví dụ `1` ở `[0, 1, 4, 5, 6]` nên `(4, 5, 6)` tốt hơn `(0, 1, 4)`.

In [19]:
def wrong_first_three(nums):
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    best = float('inf')
    for p in pos.values():
        if len(p) >= 3:
            best = min(best, 2 * (p[2] - p[0]))              # SAI: chỉ xét t = 0
    return -1 if best == float('inf') else best

nums = [1, 1, 2, 2, 1, 1, 1]
print(wrong_first_three(nums), "!=", Solution().minimumDistance(nums))
assert wrong_first_three(nums) == 8 and Solution().minimumDistance(nums) == 4

8 != 4


### Lỗi 3: quên đổi `inf` thành `-1`

Khi không có bộ ba tốt, `best` vẫn là `inf`; đề yêu cầu trả `-1`.

In [20]:
def wrong_inf(nums):
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    best = float('inf')
    for p in pos.values():
        for t in range(len(p) - 2):
            best = min(best, 2 * (p[t + 2] - p[t]))
    return best                                              # SAI: thiếu đổi inf -> -1

print(wrong_inf([1]), "!=", Solution().minimumDistance([1]))
assert wrong_inf([1]) == float('inf') and Solution().minimumDistance([1]) == -1

inf != -1


### Lỗi 4: quên điều kiện "ba giá trị bằng nhau"

Lấy ba chỉ số liên tiếp bất kỳ thì khoảng cách luôn là `4` — vô nghĩa vì giá trị chưa bằng nhau.

In [21]:
def wrong_ignore_values(nums):
    n = len(nums)
    if n < 3:
        return -1
    return min(2 * ((i + 2) - i) for i in range(n - 2))      # SAI: không kiểm tra nums[i] == nums[i+1] == nums[i+2]

print(wrong_ignore_values([1, 2, 3]), "!=", Solution().minimumDistance([1, 2, 3]))
assert wrong_ignore_values([1, 2, 3]) == 4 and Solution().minimumDistance([1, 2, 3]) == -1

4 != -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng các khoảng cách đôi một giữa ba điểm trên trục" -> luôn bằng `2 * (max - min)`; điểm giữa bị triệt tiêu.
- "Chọn `k` chỉ số cùng giá trị sao cho gần nhau nhất" -> **gom chỉ số theo giá trị** rồi dùng **cửa sổ kích thước `k`** trên danh sách chỉ số.
- Muốn tiết kiệm bộ nhớ: chỉ giữ `k - 1` vị trí gần nhất của mỗi giá trị (ở đây hai vị trí).
- Danh sách chỉ số thu được khi duyệt trái sang phải **đã sắp xếp**, không cần sort lại.
- Nhớ đổi `inf` thành `-1` ở cuối.
- Tự kiểm tra nhanh: mảng toàn số bằng nhau (`n >= 3`) cho đáp án `4`.

## 15. Biến thể của bài

1. **Minimum Distance Between Three Equal Elements II** (LeetCode 3741): cùng bài, ràng buộc lớn hơn, cần lời giải `O(n)`.
2. **Contains Duplicate II** (219): hai chỉ số cùng giá trị với khoảng cách `<= k`.
3. **Minimum Consecutive Cards to Pick Up** (2260): đoạn ngắn nhất chứa hai lá bài giống nhau.
4. **Find Closest Equal Element Queries** (3488): khoảng cách vòng giữa các chỉ số cùng giá trị.
5. **Shortest Word Distance** (243): khoảng cách nhỏ nhất giữa hai từ.
6. Tổng quát: tìm `m` chỉ số cùng giá trị với `2 * (max - min)` nhỏ nhất -> cửa sổ kích thước `m`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Ba vòng lặp | thử mọi bộ ba, tính theo định nghĩa | `O(n^3)` | `O(1)` | Oracle kiểm tra |
| Gom nhóm | `pos[v]` + cửa sổ 3 | `O(n)` | `O(n)` | Nộp bài, dễ chứng minh |
| Một lượt | nhớ hai vị trí gần nhất | `O(n)` | `O(m)` | Dữ liệu dạng luồng, tiết kiệm bộ nhớ |
| Sắp xếp | khóa `(giá trị, chỉ số)` + cửa sổ 3 | `O(n log n)` | `O(n)` | Không muốn dùng `dict` |

Nộp cách gom nhóm (hoặc một lượt nếu muốn gọn bộ nhớ).

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minimumDistance([2, 2, 2, 1, 1, 1])` ra gì?

**Bài 2 (tính tay):** với `nums = [3, 1, 3, 3, 1, 3, 1]`, liệt kê danh sách chỉ số của từng giá trị và đáp án.

**Bài 3 (điền TODO):** hoàn thành `min_span(p)` nhận danh sách chỉ số tăng dần `p` và trả về `min(p[t+2] - p[t])`, hoặc `None` nếu `len(p) < 3`.

**Bài 4 (tự viết lại):** viết `minimum_distance_k(nums, m)` tổng quát cho `m` chỉ số cùng giá trị (khoảng cách `2*(max-min)`); với `m = 3` phải trùng đáp án bài gốc.

**Bài 5 (mini challenge):** viết `best_triple(nums)` trả về **bộ ba chỉ số** `(i, j, k)` đạt khoảng cách nhỏ nhất (hoặc `None`).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minimumDistance([2, 2, 2, 1, 1, 1]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def min_span_todo(p):
    # TODO: trả về min(p[t + 2] - p[t]) hoặc None nếu p có dưới 3 phần tử
    return None

4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: giá trị `2` ở `0, 1, 2`; giá trị `1` ở `3, 4, 5`. Bài 2: giá trị `3` ở `0, 2, 3, 5`; giá trị `1` ở `1, 4, 6`.

**Mức 2 (vừa):** Bài 3: dùng `range(len(p) - 2)` và `min(..., default=None)`. Bài 4: cửa sổ `p[t + m - 1] - p[t]`. Bài 5: nhớ bộ ba `(p[t], p[t+1], p[t+2])` khi cập nhật min.

**Mức 3 (gần đáp án):** Bài 1: `4`. Bài 2: giá trị 3: `(0,2,3) -> 6`, `(2,3,5) -> 6`; giá trị 1: `(1,4,6) -> 10`; đáp án `6`. Bài 4: `2 * min(p[t + m - 1] - p[t])` với `m` chỉ số liên tiếp.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().minimumDistance([2, 2, 2, 1, 1, 1]) == 4
assert Solution().minimumDistance([3, 1, 3, 3, 1, 3, 1]) == 6

# Bài 3
def min_span(p):
    return min((p[t + 2] - p[t] for t in range(len(p) - 2)), default=None)
assert min_span([0, 2, 3, 5]) == 3 and min_span([1, 4, 6]) == 5 and min_span([4, 7]) is None

# Bài 4
def minimum_distance_k(nums, m):
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    best = None
    for p in pos.values():
        for t in range(len(p) - m + 1):
            cand = 2 * (p[t + m - 1] - p[t])
            best = cand if best is None else min(best, cand)
    return -1 if best is None else best
import random
random.seed(1)
for _ in range(300):
    n = random.randint(1, 25)
    nums = [random.randint(1, min(n, 4)) for _ in range(n)]
    assert minimum_distance_k(nums, 3) == Solution().minimumDistance(nums)
assert minimum_distance_k([1, 2, 1], 2) == 4

# Bài 5
def best_triple(nums):
    pos = defaultdict(list)
    for i, v in enumerate(nums):
        pos[v].append(i)
    best, arg = float('inf'), None
    for p in pos.values():
        for t in range(len(p) - 2):
            d = 2 * (p[t + 2] - p[t])
            if d < best:
                best, arg = d, (p[t], p[t + 1], p[t + 2])
    return arg
assert best_triple([1, 2, 1, 1, 3]) == (0, 2, 3)
assert best_triple([1, 1, 2, 3, 2, 1, 2]) == (2, 4, 6)
assert best_triple([1]) is None
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "`m` chỉ số cùng giá trị gần nhau nhất":**

```python
pos = defaultdict(list)
for i, v in enumerate(nums):
    pos[v].append(i)                      # danh sách chỉ số tăng dần
best = inf
for p in pos.values():
    for t in range(len(p) - m + 1):       # cửa sổ m phần tử liên tiếp
        best = min(best, 2 * (p[t + m - 1] - p[t]))   # m = 3 trong bài này
return -1 if best == inf else best
```

**Checklist trước khi submit:**

- [ ] Đã rút gọn khoảng cách thành `2 * (k - i)` (chỉ số giữa triệt tiêu)?
- [ ] Chỉ xét ba chỉ số **liên tiếp** trong danh sách chỉ số của cùng một giá trị?
- [ ] Giá trị xuất hiện dưới 3 lần bị bỏ qua; `range(len(p) - 2)` không gây lỗi?
- [ ] Trả `-1` (không phải `inf`) khi không có bộ ba tốt?
- [ ] Thử `[1]`, `[1, 1]`, `[1, 1, 1]`, mảng toàn một giá trị `n = 100`?
- [ ] Chữ ký đúng `minimumDistance(self, nums: List[int]) -> int`?